In [1]:
import pandas as pd

In [2]:
marks = pd.read_csv('marks_export.csv')
qa = pd.read_csv('question_answers_export.csv')

In [3]:
marks.info()

<class 'pandas.DataFrame'>
RangeIndex: 5852994 entries, 0 to 5852993
Data columns (total 12 columns):
 #   Column                       Dtype
---  ------                       -----
 0   mark_id                      int64
 1   created_at                   str  
 2   client_id                    int64
 3   client_name                  str  
 4   programme_subscription_id    int64
 5   programme_subscription_name  str  
 6   assessment_id                int64
 7   assessment_name              str  
 8   enrolment_id                 int64
 9   user_id                      int64
 10  mark_correct                 bool 
 11  unit_attempt                 int64
dtypes: bool(1), int64(7), str(4)
memory usage: 496.8 MB


In [4]:
qa.info()

<class 'pandas.DataFrame'>
RangeIndex: 5852994 entries, 0 to 5852993
Data columns (total 25 columns):
 #   Column                       Dtype
---  ------                       -----
 0   mark_id                      int64
 1   created_at                   str  
 2   client_id                    int64
 3   client_name                  str  
 4   programme_subscription_id    int64
 5   programme_subscription_name  str  
 6   assessment_id                int64
 7   assessment_name              str  
 8   enrolment_id                 int64
 9   user_id                      int64
 10  mark_correct                 bool 
 11  question_id                  int64
 12  question_position            int64
 13  question_text                str  
 14  question_type                str  
 15  answer_correct               bool 
 16  chosen_option_ids            str  
 17  chosen_option_texts          str  
 18  chosen_option_indicators     str  
 19  all_correct_option_ids       str  
 20  n_options_tot

# Understanding Our Data Format: CSV vs Parquet

Before we dive into the analysis, it's worth taking a moment to understand the type of file we're working with today and *why* it matters. You'll encounter both **CSV** and **Parquet** files constantly in data work, so building an intuition for the difference early on will save you a lot of confusion down the road.

---

## What is a CSV?

A CSV (Comma-Separated Values) file is essentially a plain text file that represents a table. If you opened one in a basic text editor, it would look exactly like this:

```
order_id,date,product,category,quantity,unit_price
1001,2024-01-05,Laptop,Electronics,1,12999
1002,2024-01-07,Desk Chair,Furniture,2,2499
```

Every row is a record, and every value is separated by a comma. It's simple, universally readable, and you can open it in Excel, Google Sheets, Notepad — anything. Think of it as the **"plain English"** of data files: great for humans, but not designed with computers in mind.

The hidden cost of that simplicity is performance. When a computer reads a CSV, it has to scan through **every row and every column**, even if you only care about one column. Imagine being handed a stack of a million printed pages and told to find one specific word — you'd have to flip through every single page.

---

## What is a Parquet?

Parquet is a smarter, more efficient way of storing the same tabular data — but designed specifically for how computers and analytical tools actually *use* data.

The key difference is **how it organises information on disk**. Instead of storing data row by row (one full record at a time), Parquet stores data **column by column**. Think of it as keeping a separate, tightly packed list for each column — one list of all names, one list of all salaries, one list of all departments, and so on.

Why does that matter? Because in data analysis, we almost never need *every* column at once. If you want to calculate the average units ordered a Parquet file lets the computer go **straight to the quantity column**, read only that, and completely ignore everything else. No wasted effort.

On top of that, Parquet applies **compression** very effectively. Because each column contains values of the same type (all numbers, or all text, etc.), compression algorithms can squeeze them down dramatically. A 1 GB CSV file might shrink to around ~100 MB as a Parquet file — a huge deal when working with cloud storage or large datasets.

Parquet also **remembers data types**. A CSV has no memory: every time you load one, your tool has to guess whether `20240115` is a number, a date, or a string. Parquet encodes the type alongside the data, so there's no guessing — and no nasty surprises where your dates get misread or your IDs turn into scientific notation.

---

## The Trade-off Worth Knowing

Parquet is not human-readable. If you opened one in a text editor, you'd see garbled binary characters — it's not meant to be read by humans directly. You need a tool like Python (pandas, polars), SQL engines (DuckDB, BigQuery), or Spark to work with it. CSV, on the other hand, can always be opened in Excel in a pinch.

So the rule of thumb is: **use CSV when you need simplicity and shareability, use Parquet when you need speed and scale.**

---

## The One-Liner

> *CSV is like a Word document — anyone can open and read it, but it's slow to process at scale. Parquet is like a database — blazing fast for analysis, but you need the right tools to open it.*

---


In [ ]:
marks.to_parquet('marks_export.parquet', index=False)
qa.to_parquet('question_answers_export.parquet', index=False)

# 🦆 What is DuckDB — and Why Are We Using It?

Now that we understand why we're working with a Parquet file instead of a CSV, there's one more tool worth understanding before we dig in: **DuckDB**. You'll see it used throughout this notebook, and once you understand what problem it solves, you'll immediately see why it's become a go-to in modern data analysis.

---

## The Problem: What Happens When Data Is Too Big for Memory?

Our dataset has close to **6 million rows**. That's not "big data" in the extreme sense, but it's large enough that loading all of it into memory at once — the way pandas normally works — is slow, wasteful, and sometimes just not possible depending on the machine you're working on.

To understand why this matters, think about the difference between two ways of looking something up in a book. The first approach is to photocopy the entire book, carry all those pages to your desk, and then search through them. The second approach is to walk to the bookshelf, open to exactly the right page, read only what you need, and walk away. Pandas (by default) is the first approach — it copies everything into memory first, *then* works with it. DuckDB is the second approach.

---

## So What Exactly Is DuckDB?

DuckDB is an **in-process analytical database** — which sounds fancy, but let's unpack it into plain English.

The "database" part means it understands SQL. You can write `SELECT`, `WHERE`, `GROUP BY`, `JOIN` — all the SQL you'd write against a traditional database like PostgreSQL or SQL Server. That's great news because it means the querying skills you build here transfer directly.

The "in-process" part is what makes it special. Unlike a traditional database (which runs as a separate server that you connect to over a network), DuckDB runs **directly inside your Python session** — no server to set up, no credentials to manage, no connection strings to debug. You just import it and start querying. It's as easy to set up as pandas, but far more powerful for analytical workloads.

The "analytical" part is the real differentiator. DuckDB is built specifically for the kind of queries data analysts run: aggregations, filters, joins across large tables, column-level operations. It uses columnar processing under the hood (similar to what we discussed with Parquet), which makes it exceptionally fast for exactly this kind of work.

---

## The Magic: Querying Parquet Without Loading It Into Memory

Here's the part that makes DuckDB particularly valuable for what we're doing today. DuckDB can **query a Parquet file directly on disk** — without ever fully loading it into memory first. It reads only the columns and rows it actually needs to answer your query, processes them efficiently, and hands you back a result.

That means instead of doing this (the pandas way):

```python
import pandas as pd

# This loads all 6 million rows into RAM immediately — slow and memory-hungry
df = pd.read_parquet("our_dataset.parquet")

# Now filter down to what we actually wanted
result = df[df["region"] == "Western Cape"].groupby("category")["revenue"].sum()
```

We can do this (the DuckDB way):

```python
import duckdb

# DuckDB reads the file on disk and only processes what the query touches
# The full 6 million rows are never loaded into memory
result = duckdb.sql("""
    SELECT
        category,
        SUM(revenue) AS total_revenue
    FROM 'our_dataset.parquet'       -- DuckDB reads directly from the file
    WHERE region = 'Western Cape'    -- Filters happen before data enters memory
    GROUP BY category
    ORDER BY total_revenue DESC
""").df()  # .df() converts the result (which IS small) into a pandas DataFrame
```

Notice that `.df()` at the end — that's perfectly fine, because the *result* of your query is small. It's only a summary table. We're not asking pandas to hold 6 million rows; we're asking it to hold maybe 20 rows of aggregated output. That's a completely reasonable use of memory.

---

## DuckDB and Parquet: A Natural Pairing

You might notice that DuckDB and Parquet feel like they were designed for each other — and that's not a coincidence. Both are built around the same columnar philosophy: only touch the data you actually need. When you run a DuckDB query against a Parquet file asking for two specific columns, DuckDB knows to skip all the other columns entirely, and Parquet's column-by-column storage makes that possible. The result is a workflow that stays fast and memory-efficient even as your data grows.

This combination — **Parquet for storage, DuckDB for querying** — is increasingly the standard approach for local analytical work, replacing the old habit of loading everything into a pandas DataFrame just to filter it down immediately afterward.

---

## How to Think About the Tools Together

A useful mental model for everything we're using today is to think of it like a professional kitchen. The **Parquet file** is the walk-in fridge — organised, efficient, and holds a huge amount of ingredients in a compressed, well-structured way. **DuckDB** is the chef — it walks into the fridge, grabs *only* the ingredients needed for the dish, and prepares them. **Python and pandas** are the plating and presentation layer — once the dish (the query result) is ready and small, we bring it out to the table for the final touches: visualisation, reporting, further analysis.

Each tool is doing the job it was actually designed for, and that's exactly what makes this stack so effective.

---

*Alright — with that foundation in place, let's start querying. You'll see all of this click into place once we run our first few cells.* 🦆

## Reading from Parquet on Disk

This is the scenario DuckDB was actually built for. Pandas has to load the **entire file into memory first**, then run the groupby. DuckDB reads only the columns it needs (`programme_subscription_id`, `programme_subscription_name`, `mark_correct`), skips everything else, and never fully loads the file at all.

Watch what happens to the timings.


In [5]:
import time
import duckdb

In [6]:
# --- Pandas: read from Parquet ---
start = time.time()

qa_parquet = pd.read_parquet('question_answers_export.parquet')
prog_accuracy_pandas_parquet = (
    qa_parquet.groupby(['programme_subscription_id', 'programme_subscription_name'])
    .agg(
        total_marks=('mark_correct', 'count'),
        correct_marks=('mark_correct', 'sum')
    )
    .reset_index()
)
prog_accuracy_pandas_parquet['pct_correct'] = (
    100.0 * prog_accuracy_pandas_parquet['correct_marks'] / prog_accuracy_pandas_parquet['total_marks']
).round(2)
prog_accuracy_pandas_parquet = prog_accuracy_pandas_parquet.sort_values('pct_correct', ascending=False).reset_index(drop=True)

pandas_parquet_elapsed = time.time() - start
print(f"⏱️  Pandas (Parquet) took: {pandas_parquet_elapsed:.4f} seconds")


⏱️  Pandas (Parquet) took: 11.8265 seconds


In [7]:
# --- DuckDB: read from Parquet ---
start = time.time()

prog_accuracy_duckdb_parquet = duckdb.sql("""
  SELECT
    programme_subscription_id,
    programme_subscription_name,
    COUNT(*) AS total_marks,
    SUM(CASE WHEN mark_correct THEN 1 ELSE 0 END) AS correct_marks,
    ROUND(100.0 * SUM(CASE WHEN mark_correct THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_correct
  FROM 'question_answers_export.parquet'
  GROUP BY programme_subscription_id, programme_subscription_name
  ORDER BY pct_correct DESC
""").df()

duckdb_parquet_elapsed = time.time() - start
print(f"⏱️  DuckDB (Parquet) took: {duckdb_parquet_elapsed:.4f} seconds")
print(f"\n📊 Result: {'DuckDB' if duckdb_parquet_elapsed < pandas_parquet_elapsed else 'Pandas'} was {max(pandas_parquet_elapsed, duckdb_parquet_elapsed) / min(pandas_parquet_elapsed, duckdb_parquet_elapsed):.1f}x faster")

print("\n--- Summary ---")
print(f"  From Parquet on disk →  Pandas: {pandas_parquet_elapsed:.4f}s  |  DuckDB: {duckdb_parquet_elapsed:.4f}s")


⏱️  DuckDB (Parquet) took: 0.5561 seconds

📊 Result: DuckDB was 21.3x faster

--- Summary ---
  From Parquet on disk →  Pandas: 11.8265s  |  DuckDB: 0.5561s


## Lesson: Use the Right Tool for the Job

In data analysis — and in software engineering more broadly — one of the most important skills isn't knowing how to use any single tool. It's knowing **which tool to reach for** and why.

Every tool we've seen today exists because someone had a specific problem that the previous tools didn't solve well enough:

- **Excel** is brilliant when you need to quickly eyeball data, share a file with someone who isn't technical, or build a simple chart for a presentation. It falls apart the moment your data has more rows than it can hold, or when you need to repeat the same analysis reliably.

- **Pandas** shines when you need to clean, reshape, and explore data in code — making your work repeatable and shareable with other developers. But it loads everything into memory at once, which becomes a problem as data grows.

- **SQL** is the language of data. When your data lives in a database and you need to ask structured questions of it, nothing beats SQL for clarity and precision. It's been around for 50 years for a reason.

- **DuckDB** steps in when your data is too large to comfortably load into memory, but you still want to write SQL and stay inside Python. It's the best of both worlds for analytical work.

The pattern you'll notice across your whole career is this: **tools are designed with specific problems in mind.** A hammer is the right tool for a nail — but not for a screw. Using pandas to process 50 million rows isn't *wrong* in the way that using a hammer on a screw is wrong, but it's slow, frustrating, and there's a better option sitting right next to it.

The engineers and analysts who are most effective aren't the ones who know the most tools — they're the ones who have a clear mental model of *what each tool is optimised for*, and the judgment to match the tool to the task.

> The goal is never to use the most impressive tool. The goal is to solve the problem well.

As you learn more tools over time, keep asking yourself: *"What problem was this built to solve?"* That question will serve you far better than memorising syntax.

In [8]:
# Clean up memory
import gc

del marks
del qa
gc.collect()

173